# Estabilidad del pipeline completo — UMAP + K-Means (Saber Pro)

Notebook independiente para responder a la observación del **editor** y del **Revisor 2** sobre la estabilidad del pipeline completo (no solo la inicialización de K-Means con el embedding de UMAP fijo). Reutiliza exactamente el mismo preprocesamiento de `pipeline_clustering_optimizado_9.ipynb`.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `Proyecto/`.
3. `Entorno de ejecución` → `Ejecutar todas`.

**La celda 5 (la del bucle `for`) ya ejecuta las 15 repeticiones automáticamente dentro de una sola ejecución** — no hace falta correr el notebook ni esa celda varias veces, el bucle interno se encarga de todo. Vas a ver el progreso impreso repetición por repetición. Tarda entre 15 y 30 minutos según los recursos que te asigne Colab.

El progreso se guarda en tu Google Drive después de cada repetición: si Colab se desconecta, vuelve a correr la misma celda y retoma donde quedó (no empieza de cero).

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment
import umap.umap_ as umap_cpu
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    df_filtrado = df_limpio[[c for c in cols_usar if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"✅ Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("⚠️  NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"✅ Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler

In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
print(f"Shape X_full: {X_full.shape}")

## 3. Análisis de estabilidad — 15 repeticiones (automático)

Esta celda **ejecuta las 15 repeticiones dentro de un solo `for`, en una sola corrida** — no hace falta correrla ni correr el notebook varias veces. En cada repetición se:
1. toma una muestra de ajuste de 80,000 filas extraída de forma **independiente** (semilla distinta),
2. se ajusta UMAP con una semilla distinta sobre esa muestra,
3. se transforma un conjunto de evaluación fijo (50,000 filas, igual en las 15 repeticiones),
4. se corre K-Means (K=8) sobre ese embedding,
5. se guardan las etiquetas resultantes.

Al final se calcula el ARI por pares entre las 15 repeticiones, la estabilidad por clúster, y una matriz de consenso.

In [ ]:
K = 8            # igual al manuscrito publicado (ajusta si tu K final es otro)
N_EVAL = 50_000  # conjunto de evaluación fijo, constante en todas las repeticiones
N_FIT = 80_000   # tamaño de la muestra de ajuste de UMAP en cada repetición
N_REPS = 15      # número de repeticiones — sube esto si tienes tiempo de sobra

OUT_DIR = '/content/drive/MyDrive/Proyecto/estabilidad_pipeline'
os.makedirs(OUT_DIR, exist_ok=True)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

n_total = X_full.shape[0]
rng_eval = np.random.default_rng(seed=0)
idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)
X_eval = X_full[idx_eval]

labels_path = os.path.join(OUT_DIR, 'labels_all.npy')
done_path = os.path.join(OUT_DIR, 'done_reps.json')

if os.path.exists(labels_path) and os.path.exists(done_path):
    labels_all = np.load(labels_path)
    done_reps = json.load(open(done_path))
    log(f"Reanudando: {len(done_reps)} repeticiones ya completas")
else:
    labels_all = np.full((N_REPS, N_EVAL), -1, dtype=int)
    done_reps = []

for r in range(N_REPS):
    if r in done_reps:
        continue
    t_r = time.time()
    seed = 100 + r

    rng_fit = np.random.default_rng(seed=seed)
    idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
    X_fit = X_full[idx_fit]

    reducer = umap_cpu.UMAP(n_components=2, random_state=seed, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
    reducer.fit(X_fit)
    emb_eval = reducer.transform(X_eval)

    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init='auto', batch_size=10_000)
    labels = km.fit_predict(emb_eval)

    labels_all[r] = labels
    done_reps.append(r)
    np.save(labels_path, labels_all)
    json.dump(done_reps, open(done_path, 'w'))
    log(f"Repetición {r+1}/{N_REPS} lista en {time.time()-t_r:.1f}s (seed={seed})")

log("✅ Todas las repeticiones completas.")

## 4. Resultados: ARI, estabilidad por clúster y matriz de consenso

In [ ]:
log("Calculando ARI por pares...")
aris = []
for i in range(N_REPS):
    for j in range(i + 1, N_REPS):
        aris.append(adjusted_rand_score(labels_all[i], labels_all[j]))
aris = np.array(aris)
log(f"ARI: media={aris.mean():.4f}  DE={aris.std():.4f}  min={aris.min():.4f}  max={aris.max():.4f}")
np.save(os.path.join(OUT_DIR, 'ari_pairs.npy'), aris)

# Alinear repeticiones contra la repetición 0 (referencia) vía Hungarian
ref = labels_all[0]
aligned = np.zeros_like(labels_all)
aligned[0] = ref
for r in range(1, N_REPS):
    cost = np.zeros((K, K))
    for a in range(K):
        for b in range(K):
            cost[a, b] = -np.sum((ref == a) & (labels_all[r] == b))
    row_ind, col_ind = linear_sum_assignment(cost)
    mapping = {b: a for a, b in zip(row_ind, col_ind)}
    aligned[r] = np.array([mapping.get(lab, -1) for lab in labels_all[r]])
np.save(os.path.join(OUT_DIR, 'labels_aligned.npy'), aligned)

agree = (aligned[1:] == ref[None, :]).mean(axis=0)
cluster_stability = {}
for c in range(K):
    mask = ref == c
    cluster_stability[c] = {
        'n_puntos': int(mask.sum()),
        'tasa_acuerdo_media': float(agree[mask].mean()) if mask.sum() > 0 else None,
    }
    log(f"Cluster {c}: n={mask.sum():,}  acuerdo medio={cluster_stability[c]['tasa_acuerdo_media']:.3f}")

# Matriz de consenso sobre una submuestra de 1,500 puntos
rng_co = np.random.default_rng(seed=7)
idx_co = rng_co.choice(N_EVAL, size=1500, replace=False)
sub = aligned[:, idx_co]
co_matrix = np.zeros((1500, 1500), dtype=np.float32)
for r in range(N_REPS):
    lab = sub[r]
    co_matrix += (lab[:, None] == lab[None, :])
co_matrix /= N_REPS
np.save(os.path.join(OUT_DIR, 'co_matrix.npy'), co_matrix)
np.save(os.path.join(OUT_DIR, 'ref_labels_co.npy'), ref[idx_co])

summary = {'K': K, 'N_EVAL': N_EVAL, 'N_FIT': N_FIT, 'N_REPS': N_REPS,
           'ari_mean': float(aris.mean()), 'ari_sd': float(aris.std()),
           'ari_min': float(aris.min()), 'ari_max': float(aris.max()),
           'cluster_stability': cluster_stability}
json.dump(summary, open(os.path.join(OUT_DIR, 'summary.json'), 'w'), indent=2)
log(f"✅ Resultados guardados en {OUT_DIR}")

## 5. Figura: distribución de ARI + matriz de consenso

In [ ]:
order = np.argsort(ref[idx_co], kind='stable')
co_ordered = co_matrix[order][:, order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), dpi=150,
                                gridspec_kw={"width_ratios": [1, 1.15]})

ax1.hist(aris, bins=18, color="#3B6FA0", edgecolor="white", linewidth=0.6)
ax1.axvline(aris.mean(), color="#B33F3F", linewidth=1.6, linestyle="--",
            label=f"Media = {aris.mean():.3f}")
ax1.axvline(0.69, color="#555555", linewidth=1.4, linestyle=":",
            label="ARI reportado en el manuscrito = 0.69")
ax1.set_xlabel("ARI por par de repeticiones")
ax1.set_ylabel("N.º de pares")
ax1.set_title(f"Estabilidad del pipeline completo (n={N_REPS} repeticiones)", fontsize=10)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.legend(fontsize=7.5, frameon=False, loc="upper left")

im = ax2.imshow(co_ordered, cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
ax2.set_title("Matriz de consenso (n=1,500)\nordenada por clúster de referencia", fontsize=10)
ax2.set_xticks([]); ax2.set_yticks([])
cbar = plt.colorbar(im, ax=ax2, shrink=0.85)
cbar.set_label("Prob. de co-asignación", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_estabilidad_pipeline.png'), dpi=200,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura guardada en Drive")